In [5]:
import json

class ClaudeJsonlBuilder:
    def __init__(self, output_path: str, default_max_tokens: int = 1024, default_temperature: float = 0.5):
        self.output_path = output_path
        self.default_max_tokens = default_max_tokens
        self.default_temperature = default_temperature
        self.data = []

    def build_request(
        self,
        record_id: str,
        prompt: str,
        max_token: int = None,
        temperature: float = None,
        enable_thinking: bool = False,
        thinking_budget_tokens: int = 2000
    ) -> dict:
        """
        Tạo 1 request JSON theo định dạng Claude Bedrock.
        """
        model_input = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_token if max_token is not None else self.default_max_tokens,
            "temperature": temperature if temperature is not None else self.default_temperature,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }
            ]
        }

        if enable_thinking:
            model_input["temperature"] = 1  # Override nếu thinking bật
            model_input["thinking"] = {
                "type": "enabled",
                "budget_tokens": thinking_budget_tokens
            }

        return {
            "recordId": record_id,
            "modelInput": model_input
        }

    def add_prompt(self, record_id: str, prompt: str, **kwargs):
        """
        Thêm một prompt vào danh sách request.
        """
        request = self.build_request(record_id, prompt, **kwargs)
        self.data.append(request)

    def write_jsonl(self):
        """
        Ghi tất cả các request đã thêm vào file JSONL.
        """
        with open(self.output_path, 'w', encoding='utf-8') as f:
            for item in self.data:
                line = json.dumps(item, ensure_ascii=False)
                f.write(line + '\n')

In [6]:

# Khởi tạo builder với đường dẫn file xuất
builder = ClaudeJsonlBuilder(output_path="claude_requests.jsonl")

# Thêm các prompt
builder.add_prompt("CALL0001", "Summarize this transcript...")
builder.add_prompt("CALL0002", "What are the key issues?")
builder.add_prompt("CALL0003", "List action items mentioned by the customer.")

# Xuất ra file .jsonl
builder.write_jsonl()
